# Notebook 6: Convolutional Neural Networks — Introduction
**Estimated time:** 35-45 min
**Prerequisites:** Notebooks 1-5

---
## What you will learn
1. Why convolutions for images — and why not flat linear layers
2. How the convolution operation works (with worked examples)
3. `nn.Conv2d` parameters explained
4. Pooling layers (MaxPool, AvgPool)
5. Building your first CNN block
6. Visualizing feature maps

## 1. The Problem with Flat Linear Layers on Images

A 224×224 RGB image has 224 × 224 × 3 = **150,528 input features**.
A single hidden layer with 1000 neurons would need 150 million weights — just for layer 1!

But that's not even the real problem. Linear layers:
- Don't know that nearby pixels are related
- Don't understand that patterns (like edges) can appear anywhere in the image
- Can't share knowledge: if they learn "cat ear" in the top-left, they don't know it could be top-right

**Convolutions** solve all this by:
1. Using small **filters** that look at local regions (not the whole image)
2. **Sharing weights** — the same filter scans every position in the image
3. **Hierarchy** — early layers learn edges, later layers learn complex shapes

## 2. How Convolution Works

**Think of a convolution like a flashlight scanning a dark room.**
The flashlight (filter/kernel) is small — say 3×3 pixels. You slide it over every patch of the image.
At each position, you measure how strongly the flashlight pattern matches what's there.
This produces a new image (feature map) showing where the pattern appears.

Mathematically, for each position (i, j):
```
output[i, j] = sum(filter * image_patch[i:i+k, j:j+k])
```

Let's see this with actual images:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Create a simple 7x7 test image (a cross/plus shape)
img = torch.zeros(7, 7)
img[3, :] = 1.0   # horizontal bar
img[:, 3] = 1.0   # vertical bar

print('Our test image:')
print(img)

plt.figure(figsize=(3, 3))
plt.imshow(img, cmap='gray', vmin=0, vmax=1)
plt.title('Test Image (cross)')
plt.colorbar()
plt.show()

In [ ]:
# Define some hand-crafted filters
filters = {
    'Edge (horizontal)': torch.tensor([[-1., -1., -1.],
                                        [ 2.,  2.,  2.],
                                        [-1., -1., -1.]]),
    'Edge (vertical)':   torch.tensor([[-1.,  2., -1.],
                                        [-1.,  2., -1.],
                                        [-1.,  2., -1.]]),
    'Blur (avg)':        torch.ones(3, 3) / 9.0,
    'Sharpen':           torch.tensor([[ 0., -1.,  0.],
                                        [-1.,  5., -1.],
                                        [ 0., -1.,  0.]]),
}

# Apply each filter using F.conv2d
# F.conv2d expects (batch, channels, H, W) shaped tensors
img_4d = img.unsqueeze(0).unsqueeze(0)   # (1, 1, 7, 7)

fig, axes = plt.subplots(1, len(filters) + 1, figsize=(16, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for ax, (name, filt) in zip(axes[1:], filters.items()):
    # Reshape filter to (out_ch, in_ch, H, W)
    filt_4d = filt.unsqueeze(0).unsqueeze(0)
    output  = F.conv2d(img_4d, filt_4d, padding=1)   # padding=1 keeps size
    out_img = output.squeeze().detach()
    ax.imshow(out_img, cmap='RdBu', vmin=-3, vmax=3)
    ax.set_title(name, fontsize=10)
    ax.axis('off')

plt.suptitle('Hand-crafted filters applied to the cross image', fontsize=12)
plt.tight_layout()
plt.show()

print('CNN training learns these filters automatically from data!')

## 3. `nn.Conv2d` Parameters

```python
nn.Conv2d(
    in_channels,   # number of input channels  (1 for grayscale, 3 for RGB)
    out_channels,  # number of filters to learn (= number of output feature maps)
    kernel_size,   # filter size: 3 means 3x3, or (3, 5) for different H/W
    stride=1,      # how many pixels to skip between positions (1 = no skip)
    padding=0,     # how many zeros to add around the border
    bias=True      # add a learnable bias per filter
)
```

**Output size formula:**
```
H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
```

**Common configuration:** `kernel_size=3, stride=1, padding=1` → output same size as input.

In [ ]:
# Explore Conv2d output shapes
def conv_output_size(in_size, kernel, stride=1, padding=0):
    return (in_size + 2 * padding - kernel) // stride + 1

configs = [
    (28, 3, 1, 0),   # typical: in=28, k=3, stride=1, pad=0
    (28, 3, 1, 1),   # with padding=1: keeps size
    (28, 3, 2, 0),   # stride=2: halves size
    (224, 7, 2, 3),  # ResNet first layer
]

print(f'{"in_size":>8} {"kernel":>8} {"stride":>8} {"padding":>8} {"out_size":>10}')
print('-' * 50)
for (s, k, st, p) in configs:
    out = conv_output_size(s, k, st, p)
    print(f'{s:>8} {k:>8} {st:>8} {p:>8} {out:>10}')

# Actual Conv2d example
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(4, 3, 28, 28)   # batch=4, channels=3, 28x28
out = conv(x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')   # (4, 16, 28, 28)
print(f'Learned filters: {conv.weight.shape}')  # (16, 3, 3, 3)

## 4. Pooling Layers

**Pooling is like summarizing a paragraph into one sentence.**
It reduces the spatial size of feature maps, making the network faster and more translation-invariant.

**MaxPool:** takes the max in each window — keeps the strongest activations
**AvgPool:** takes the average — smoother summary

Most CNNs use MaxPool.

In [ ]:
x = torch.tensor([[1., 2., 3., 4.],
                   [5., 6., 7., 8.],
                   [1., 2., 1., 2.],
                   [3., 4., 3., 4.]])
x = x.unsqueeze(0).unsqueeze(0)  # (1, 1, 4, 4)

max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

max_out = max_pool(x)
avg_out = avg_pool(x)

print('Input (4x4):')
print(x.squeeze())
print('MaxPool2d output (2x2):')
print(max_out.squeeze())
print('AvgPool2d output (2x2):')
print(avg_out.squeeze())

## 5. A Complete CNN Block

A standard CNN block = Conv → BatchNorm → ReLU → (optional Pooling)

This is the building block you'll stack to make a full CNN.

In [ ]:
class CNNBlock(nn.Module):
    '''Standard conv block: Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d (optional)'''

    def __init__(self, in_ch, out_ch, kernel=3, pool=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel, padding=kernel // 2)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(self.relu(self.bn(self.conv(x))))


# Build a small CNN for 28x28 grayscale images (like MNIST)
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            CNNBlock(1, 16,  pool=True),   # (B, 16, 14, 14)
            CNNBlock(16, 32, pool=True),   # (B, 32, 7, 7)
            CNNBlock(32, 64, pool=False),  # (B, 64, 7, 7)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),        # (B, 64, 1, 1) — works for any input size
            nn.Flatten(),                   # (B, 64)
            nn.Linear(64, num_classes)      # (B, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SmallCNN()
x = torch.randn(8, 1, 28, 28)
out = model(x)
print('Input :', x.shape)
print('Output:', out.shape)
print('Params:', sum(p.numel() for p in model.parameters()))

## 6. Visualizing Feature Maps

Let's pass a real image through our CNN and visualize what each filter responds to.

In [ ]:
from torchvision.datasets import MNIST
import torchvision.transforms as T

# Load one MNIST image
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
mnist = MNIST('./data', train=True, download=True, transform=transform)
img, label = mnist[0]
print(f'Image shape: {img.shape}, Label: {label}')

# Pass through first conv layer only
first_conv = nn.Conv2d(1, 16, kernel_size=3, padding=1)
img_batch = img.unsqueeze(0)   # add batch dim: (1, 1, 28, 28)
with torch.no_grad():
    feature_maps = first_conv(img_batch)

print(f'Feature maps shape: {feature_maps.shape}')  # (1, 16, 28, 28)

# Plot: original + 16 feature maps
fig = plt.figure(figsize=(14, 4))

# Original
ax = fig.add_subplot(2, 9, 1)
ax.imshow(img.squeeze(), cmap='gray')
ax.set_title('Original', fontsize=9)
ax.axis('off')

# 16 feature maps (first 8)
for i in range(8):
    ax = fig.add_subplot(2, 9, i + 2)
    fmap = feature_maps[0, i].numpy()
    ax.imshow(fmap, cmap='viridis')
    ax.set_title(f'Filter {i+1}', fontsize=8)
    ax.axis('off')

# 8 filter weights
for i in range(8):
    ax = fig.add_subplot(2, 9, i + 11)
    w = first_conv.weight[i, 0].detach().numpy()
    ax.imshow(w, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(f'Weight {i+1}', fontsize=8)
    ax.axis('off')

plt.suptitle('Top: Feature maps | Bottom: Filter weights (random init)', fontsize=11)
plt.tight_layout()
plt.show()

---
## YOUR TURN — Mini Project

**Task:** Apply hand-crafted edge detection filters to a real MNIST image.

**Steps:**
1. Load 5 different MNIST images (digits 0-9, pick variety)
2. Define these 3 filters manually as tensors:
   - Horizontal edge: `[[-1,-1,-1],[2,2,2],[-1,-1,-1]]`
   - Vertical edge: `[[-1,2,-1],[-1,2,-1],[-1,2,-1]]`
   - Diagonal: `[[2,-1,-1],[-1,2,-1],[-1,-1,2]]`
3. Apply each filter to each image using `F.conv2d(img, filter, padding=1)`
4. Display in a grid: rows = images, cols = (original + 3 filtered)
5. **Observation question:** Which filter responds most strongly to digit "1"? Why?

In [ ]:
# YOUR CODE HERE
# 1. Load 5 MNIST images

# 2. Define 3 filters

# 3. Apply filters

# 4. Display grid

# 5. Answer: which filter responds to '1' and why?